# American Options - Python Case Study

## Objective

The objective of this case study is to turn the classroom note into a runnable quant workflow. By the end, a student should be able to:

- identify the state variables used by the model;
- connect the pricing or risk formula to code;
- run the base case with a small market dataset;
- perform one controlled what-if experiment;
- interpret the result with model-risk and implementation-risk discipline.

This notebook is intentionally written as a miniature case-study chapter. Read the markdown first, then run the cells, then change one input at a time.

## Business Context

We study an American put with possible early exercise. The desk observes or controls spot tree, exercise payoff, continuation value, stopping time, and discount factor. The practical question is how to compute optimal stopping recursion in a way that is transparent enough for front-office pricing, market-risk review, and interview-style implementation.

This matters because a number alone is not enough. A quant must explain the model assumptions, the measure, the discounting convention, the numerical method, and the diagnostic checks.

## Core Mathematical Object

The central relation used in this case study is:

$$V_t=\max\left(g(S_t),e^{-r\Delta t}\mathrm{E}^{\mathrm{Q}}[V_{t+\Delta t}\mid\mathcal{F}_t]\right)$$

The formula should be read together with the information set. Conditional expectations are always conditional on what is known at the valuation or decision time. When a measure change appears, the payoff is unchanged; the probabilities or numeraire are changed.

## Case Study Design

We will use a small reproducible dataset:

- `american_options_params.csv` contains scalar assumptions such as spot, strike, volatility, maturity, path count, and model parameters.
- `american_options_market.csv` contains the market quotes, scenarios, or grid points used by this topic.
- `american_options_case_study.py` contains the reusable routines so the notebook remains readable.

The implementation goal is not to hide the math inside a library. The goal is to keep each modelling step visible enough that a student can audit the calculation.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import american_options_case_study as case

PARAMS_FILE = "american_options_params.csv"
MARKET_FILE = "american_options_market.csv"
CASE_NAME = "american_lsm"

params_table = pd.read_csv(PARAMS_FILE)
market = pd.read_csv(MARKET_FILE)

print("Parameter table")
display(params_table)

print("Market/scenario data")
display(market.head(12))

## Assumptions

Before running any pricing or risk routine, state the assumptions explicitly. This is the habit that prevents most silent implementation errors.

In this notebook:

- inputs are deterministic at time zero unless the case study says otherwise;
- simulated paths are generated under the pricing measure used by the routine;
- discounting is applied after the appropriate conditional expectation or pathwise payoff is formed;
- random seeds are fixed only to make the teaching example reproducible;
- all prices and risk numbers are examples, not live market quotes.

In [ ]:
params = params_table.set_index("parameter")["value"].astype(float)

paths = case.simulate_gbm(
    S0=float(params.get("S0", 100.0)),
    r=float(params.get("r", 0.04)),
    q=float(params.get("q", 0.01)),
    sigma=float(params.get("sigma", 0.22)),
    T=float(params.get("T", 1.0)),
    steps=int(params.get("steps", 64)),
    paths=int(params.get("paths", 12000)),
    seed=7,
)

print("Simulated path matrix shape:", paths.shape)
print("First path, first five observations:", np.round(paths[0, :5], 4))

## Base-Case Run

The next cell runs the topic-specific routine:

`case.run_topic_case("american_lsm", params, market, paths)`

Read the outputs as a calculation trail. Tables show intermediate quantities; scalar outputs are the final price, risk number, calibration diagnostic, or estimator summary.

In [ ]:
outputs = case.run_topic_case("american_lsm", params, market, paths)
outputs["bsm_call_reference"] = case.black_scholes_call(
    float(params.get("S0", 100.0)),
    float(params.get("K", 100.0)),
    float(params.get("r", 0.04)),
    float(params.get("q", 0.01)),
    float(params.get("sigma", 0.22)),
    float(params.get("T", 1.0)),
)

for key, value in outputs.items():
    print("\n" + key)
    print("-" * len(key))
    if isinstance(value, pd.DataFrame):
        display(value.round(6))
    elif isinstance(value, np.ndarray):
        print(np.round(value, 6))
    else:
        print(value)

## Interpretation

The base-case output should be interpreted through the topic objective:

- Instrument: an American put with possible early exercise.
- Main state variables: spot tree, exercise payoff, continuation value, stopping time, and discount factor.
- Main sensitivity or risk idea: early-exercise boundary and local delta around the boundary.
- Main implementation risk: perfect-foresight exercise decisions in Monte Carlo.

The first reading of the output should answer: does the sign make sense, are the units clear, and is the magnitude plausible?

## What-If Experiment

A useful notebook should let the student test intuition. The next cell changes volatility and maturity while keeping the rest of the setup fixed. The exact interpretation differs by topic, but the workflow is the same: rerun the model, compare outputs, and explain the direction of change.

In [ ]:
def run_with(overrides=None, seed=19):
    overrides = overrides or {}
    p = params.copy()
    for key, value in overrides.items():
        p.loc[key] = value
    new_paths = case.simulate_gbm(
        S0=float(p.get("S0", 100.0)),
        r=float(p.get("r", 0.04)),
        q=float(p.get("q", 0.01)),
        sigma=float(p.get("sigma", 0.22)),
        T=float(p.get("T", 1.0)),
        steps=int(p.get("steps", 64)),
        paths=int(p.get("paths", 12000)),
        seed=seed,
    )
    return case.run_topic_case("american_lsm", p, market, new_paths)

base = outputs
stress = run_with({"sigma": float(params.get("sigma", 0.22)) * 1.25, "T": float(params.get("T", 1.0)) * 1.5})

comparison_rows = []
for key in sorted(set(base) & set(stress)):
    if isinstance(base[key], (int, float, np.floating)) and isinstance(stress[key], (int, float, np.floating)):
        comparison_rows.append({"metric": key, "base": float(base[key]), "stress": float(stress[key]), "change": float(stress[key] - base[key])})

if comparison_rows:
    display(pd.DataFrame(comparison_rows).round(6))
else:
    print("This topic returns table outputs. Inspect the base and stress tables directly.")

## Visual Diagnostic

The plot is a model diagnostic. It is not a proof, but it catches many implementation errors: exploding paths, wrong scale, non-positive values where positivity is required, or an output that does not react to volatility or maturity.

In [ ]:
case.plot_case_study(show=True)

## Concept Checks

- What information is available when the exercise decision is made?
- Why does Longstaff-Schwartz regress continuation value only on in-the-money paths?
- How can perfect foresight bias enter a simulation?

## Conclusion

American option pricing is an optimal-stopping problem. The key implementation discipline is that the exercise rule can use only information available at the exercise date.

Before treating the result as usable, the student should be able to explain the objective, reproduce the base-case number, describe the what-if result, and name the most important implementation risk.

## Extensions

Try these after running the notebook once:

- Double the number of paths and check whether the output stabilizes.
- Halve the time step and compare the change with the Monte Carlo sampling error.
- Change one market quote or scenario and explain which output is most sensitive.
- Replace the default random seed and check whether the conclusion survives.
- Add a benchmark calculation or limiting case where the answer is known.